In [1]:
!pip install -q transformers datasets accelerate evaluate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00


In [8]:
from datasets import load_dataset

# Load the modern, professional Biomed NER dataset
# It's script-free, so it works perfectly in 2026
dataset = load_dataset("knowledgator/biomed_NER")

# Verify it works immediately
print(f"Success! Train rows: {len(dataset['train'])}")
print(f"Sample: {dataset['train'][0]['text'][:100]}...")

README.md: 0.00B [00:00, ?B/s]

train.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4840 [00:00<?, ? examples/s]

Success! Train rows: 4840
Sample: Weed seed inactivation in soil mesocosms via biosolarization with mature compost and tomato processi...


In [9]:
from transformers import AutoTokenizer
import numpy as np

# We use DistilRoBERTa because it's fast and efficient for "Edge Deployment"
model_id = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_id, add_prefix_space=True)

def preprocess_function(examples):
    # Tokenize the text
    tokenized_inputs = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128, return_offsets_mapping=True)

    labels = []
    for i, entities in enumerate(examples["entities"]):
        # Get the mapping from tokens to character positions
        offset_mapping = tokenized_inputs.offset_mapping[i]
        # Initialize labels as 0 (Outside)
        label_ids = [0] * len(tokenized_inputs["input_ids"][i])

        for entity in entities:
            start, end = entity["start"], entity["end"]

            for idx, (token_start, token_end) in enumerate(offset_mapping):
                # Skip special tokens (where start/end are 0)
                if token_start == token_end:
                    label_ids[idx] = -100 # Ignore in loss calculation
                    continue

                # Check if the token falls within the entity range
                if token_start >= start and token_end <= end:
                    # B-ENTITY if it's the start, I-ENTITY if it's a continuation
                    if token_start == start:
                        label_ids[idx] = 1 # B-ENTITY
                    else:
                        label_ids[idx] = 2 # I-ENTITY

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Map the function over the dataset
tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/4840 [00:00<?, ? examples/s]

In [18]:
from transformers import TrainingArguments, Trainer
# Split the training data: 90% for training, 10% for validation
split_dataset = tokenized_dataset["train"].train_test_split(test_size=0.1)

# Now we have a 'train' and a 'test' key in our new split_dataset
train_data = split_dataset["train"]
valid_data = split_dataset["test"]

print(f"New Train size: {len(train_data)}")
print(f"New Validation size: {len(valid_data)}")
training_args = TrainingArguments(
    output_dir="./biomed_ner_results",
    eval_strategy="epoch",       # Changed from evaluation_strategy
    save_strategy="epoch",       # Keep these matched
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True, # Pro-move for your resume
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=valid_data,
    processing_class=tokenizer, # Changed from tokenizer=tokenizer
    data_collator=data_collator,
)

print("Final API alignment complete. Training starting...")
trainer.train()

New Train size: 4356
New Validation size: 484
Final API alignment complete. Training starting...


Epoch,Training Loss,Validation Loss
1,0.392716,0.463218
2,0.384874,0.451997
3,0.351955,0.450238


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=819, training_loss=0.4256939768645644, metrics={'train_runtime': 236.1714, 'train_samples_per_second': 55.333, 'train_steps_per_second': 3.468, 'total_flos': 426851395660800.0, 'train_loss': 0.4256939768645644, 'epoch': 3.0})

In [19]:
# Save the final model weights and configuration
trainer.save_model("./biomed_ner_final")
tokenizer.save_pretrained("./biomed_ner_final")

# Zip it so you can download it to your laptop
import shutil
shutil.make_archive("biomed_ner_model", 'zip', "./biomed_ner_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

'/content/biomed_ner_model.zip'

In [22]:
import shutil
from google.colab import files

# 1. Zip the folder we saved earlier
shutil.make_archive("biomed_ner_final", 'zip', "/content/biomed_ner_results")

# 2. Trigger a browser download to your laptop
files.download("biomed_ner_final.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>